In [1]:
import os
import numpy as np
import pandas as pd
import mne
from scipy import signal
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
mne.set_log_level('ERROR')

In [2]:
# Rutas
DATA_DIR = '/home/manu/TFG2/TUH_EEG/eval'
OUTPUT_DIR = '/home/manu/TFG2/EEG-Gold-Standard-main/ConnectivityMatrices_TUH'

# Parámetros
FS = 256  # Frecuencia de muestreo
FREQ_BAND = (8, 12)  # Banda Alpha
WINDOW_DURATION = 2.0  # segundos por ventana
WINDOW_SAMPLES = int(WINDOW_DURATION * FS)  # 512 muestras
MAX_BCKG_PER_FILE = 10  # Máximo de ventanas background por archivo (balance)

# Canales EEG 10-20 estándar (19 canales)
EEG_CHANNELS_TARGET = [
    'FP1', 'FP2',
    'F3', 'F4', 'F7', 'F8', 'FZ',
    'C3', 'C4', 'CZ',
    'P3', 'P4', 'PZ',
    'O1', 'O2',
    'T3', 'T4', 'T5', 'T6'
]

print(f"Canales objetivo: {len(EEG_CHANNELS_TARGET)}")
print(f"Ventana: {WINDOW_DURATION}s = {WINDOW_SAMPLES} samples")
print(f"Banda: {FREQ_BAND[0]}-{FREQ_BAND[1]} Hz (Alpha)")

Canales objetivo: 19
Ventana: 2.0s = 512 samples
Banda: 8-12 Hz (Alpha)


In [3]:
# Listar todos los pacientes
patients = sorted(os.listdir(DATA_DIR))
print(f"Total pacientes: {len(patients)}")
print(f"Primeros 5: {patients[:5]}")

Total pacientes: 43
Primeros 5: ['aaaaaaaq', 'aaaaaamc', 'aaaaaarq', 'aaaaaefp', 'aaaaafcf']


In [4]:
# División train/test (cross-subject)
# 43 pacientes -> 34 train, 9 test (80/20)
np.random.seed(42)
patients_shuffled = list(np.random.permutation(patients))

train_patients = patients_shuffled[:34]
test_patients = patients_shuffled[34:]

print(f"Train: {len(train_patients)} pacientes")
print(f"Test: {len(test_patients)} pacientes")

Train: 34 pacientes
Test: 9 pacientes


In [ ]:
def normalize_channel_name(ch_name):
    """Limpia el nombre del canal: 'EEG FP1-REF' -> 'FP1'"""
    name = ch_name.replace('EEG ', '').replace('-REF', '').replace('-LE', '')
    return name.upper()


def get_channel_indices(raw, target_channels):
    """Obtiene los índices de los canales objetivo."""
    available = {normalize_channel_name(ch): i for i, ch in enumerate(raw.ch_names)}
    indices = []
    found = []
    for ch in target_channels:
        if ch in available:
            indices.append(available[ch])
            found.append(ch)
    return indices, found


def compute_coherence_matrix(data, fs, freq_band):
    """Calcula matriz de coherencia cuadrada."""
    n_channels = data.shape[0]
    coh_matrix = np.zeros((n_channels, n_channels))
    
    for i in range(n_channels):
        for j in range(i, n_channels):
            if i == j:
                coh_matrix[i, j] = 1.0
            else:
                f, Cxy = signal.coherence(data[i], data[j], fs=fs, nperseg=min(256, len(data[i])))
                freq_mask = (f >= freq_band[0]) & (f <= freq_band[1])
                coh_value = np.mean(Cxy[freq_mask]) if np.any(freq_mask) else 0.0
                coh_matrix[i, j] = coh_value
                coh_matrix[j, i] = coh_value
    return coh_matrix


def select_channels(n_target, total_channels):
    """Selecciona canales equiespaciados."""
    if n_target >= total_channels:
        return list(range(total_channels))
    indices = np.linspace(0, total_channels - 1, n_target, dtype=int)
    return indices.tolist()


def parse_csv_bi(csv_path):
    """
    Parsea archivo .csv_bi y devuelve lista de eventos.
    """
    events = []
    with open(csv_path, 'r') as f:
        for line in f:
            if line.startswith('#') or line.startswith('channel'):
                continue
            parts = line.strip().split(',')
            if len(parts) >= 4:
                start = float(parts[1])
                stop = float(parts[2])
                label = parts[3]
                events.append((start, stop, label))
    return events

In [ ]:
def process_edf_file(edf_path, csv_bi_path, ch_indices):
    """Procesa un archivo EDF y devuelve matrices."""
    seiz_matrices = []
    bckg_matrices = []
    
    try:
        raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
        if raw.info['sfreq'] != FS:
            raw.resample(FS, verbose=False)
        
        data = raw.get_data()
        
        if len(ch_indices) < 19:
            return [], []
        
        data = data[ch_indices]
        n_samples = data.shape[1]
        duration = n_samples / FS
        
        # Leer eventos de seizure
        events = parse_csv_bi(csv_bi_path)
        seiz_intervals = [(s, e) for s, e, l in events if l == 'seiz']
        
        # Si hay seizures, generar ventanas dentro de ellos
        seiz_windows = []
        for s, e in seiz_intervals:
            start_sample = int(s * FS)
            stop_sample = int(e * FS)
            for win_start in range(start_sample, stop_sample - WINDOW_SAMPLES, WINDOW_SAMPLES):
                if win_start + WINDOW_SAMPLES <= n_samples:
                    seiz_windows.append(data[:, win_start:win_start + WINDOW_SAMPLES])
        
        # Background: lo que NO sea seizure
        # Crear máscara de seizure
        is_seizure = np.zeros(n_samples, dtype=bool)
        for s, e in seiz_intervals:
            start_sample = int(s * FS)
            stop_sample = min(int(e * FS), n_samples)
            is_seizure[start_sample:stop_sample] = True
        
        # Generar ventanas de background (donde NO hay seizure)
        bckg_windows = []
        for win_start in range(0, n_samples - WINDOW_SAMPLES, WINDOW_SAMPLES):
            # Si NINGUNA muestra de la ventana es seizure
            if not np.any(is_seizure[win_start:win_start + WINDOW_SAMPLES]):
                bckg_windows.append(data[:, win_start:win_start + WINDOW_SAMPLES])
        
        # Balance: limitar background
        if len(bckg_windows) > MAX_BCKG_PER_FILE:
            indices = np.random.choice(len(bckg_windows), MAX_BCKG_PER_FILE, replace=False)
            bckg_windows = [bckg_windows[i] for i in indices]
        
        # Calcular matrices de coherencia
        for window in seiz_windows:
            mat = compute_coherence_matrix(window, FS, FREQ_BAND)
            seiz_matrices.append(mat)
        
        for window in bckg_windows:
            mat = compute_coherence_matrix(window, FS, FREQ_BAND)
            bckg_matrices.append(mat)
    
    except Exception as e:
        pass
    
    return seiz_matrices, bckg_matrices

In [7]:
def find_edf_files(patient_dir):
    """Encuentra todos los archivos EDF de un paciente."""
    edf_files = []
    for root, dirs, files in os.walk(patient_dir):
        for f in files:
            if f.endswith('.edf'):
                edf_path = os.path.join(root, f)
                csv_bi_path = edf_path.replace('.edf', '.csv_bi')
                if os.path.exists(csv_bi_path):
                    edf_files.append((edf_path, csv_bi_path))
    return edf_files

In [8]:
for split in ['Training', 'Test']:
    for n_channels in ['8', '16']:
        for label in ['bckg', 'seiz']:
            path = os.path.join(OUTPUT_DIR, split, n_channels, label)
            os.makedirs(path, exist_ok=True)

print(f"Estructura creada en: {OUTPUT_DIR}")
print("  Training/{8,16}/{bckg,seiz}/")
print("  Test/{8,16}/{bckg,seiz}/")

Estructura creada en: /home/manu/TFG2/EEG-Gold-Standard-main/ConnectivityMatrices_TUH
  Training/{8,16}/{bckg,seiz}/
  Test/{8,16}/{bckg,seiz}/


In [9]:
def process_patients(patient_list, split_name):
    """Procesa una lista de pacientes y guarda las matrices."""
    counts = {ch: {'bckg': 0, 'seiz': 0} for ch in [8, 16]}
    
    for patient_id in tqdm(patient_list, desc=split_name):
        patient_dir = os.path.join(DATA_DIR, patient_id)
        edf_files = find_edf_files(patient_dir)
        
        for edf_path, csv_bi_path in edf_files:
            # Leer una vez para obtener canales
            try:
                raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
                ch_indices, _ = get_channel_indices(raw, EEG_CHANNELS_TARGET)
            except:
                continue
            
            # Procesar archivo (devuelve matrices de 19 canales)
            seiz_mats, bckg_mats = process_edf_file(edf_path, csv_bi_path, ch_indices)
            
            if len(seiz_mats) == 0 and len(bckg_mats) == 0:
                continue
            
            # Para cada config de canales (8, 16)
            for n_ch in [8, 16]:
                ch_subset = select_channels(n_ch, 19)
                
                # Guardar seiz
                for idx, mat in enumerate(seiz_mats):
                    sub_mat = mat[np.ix_(ch_subset, ch_subset)]
                    fname = f"{patient_id}_{os.path.basename(edf_path).replace('.edf', '')}_seiz_{idx:03d}.npy"
                    path = os.path.join(OUTPUT_DIR, split_name, str(n_ch), 'seiz', fname)
                    np.save(path, sub_mat)
                    counts[n_ch]['seiz'] += 1
                
                # Guardar bckg
                for idx, mat in enumerate(bckg_mats):
                    sub_mat = mat[np.ix_(ch_subset, ch_subset)]
                    fname = f"{patient_id}_{os.path.basename(edf_path).replace('.edf', '')}_bckg_{idx:03d}.npy"
                    path = os.path.join(OUTPUT_DIR, split_name, str(n_ch), 'bckg', fname)
                    np.save(path, sub_mat)
                    counts[n_ch]['bckg'] += 1
    
    return counts

In [10]:
# Procesar Training
print("="*60)
print("GENERANDO TRAINING")
print("="*60)
train_counts = process_patients(train_patients, 'Training')

print("\n--- TRAINING ---")
for ch in [16, 8]:
    print(f"{ch} canales: bckg={train_counts[ch]['bckg']}, seiz={train_counts[ch]['seiz']}")

GENERANDO TRAINING


Training:   0%|          | 0/34 [00:00<?, ?it/s]

Training: 100%|██████████| 34/34 [2:28:06<00:00, 261.37s/it]  


--- TRAINING ---
16 canales: bckg=7482, seiz=9383
8 canales: bckg=7482, seiz=9383


In [11]:
# Procesar Test
print("="*60)
print("GENERANDO TEST")
print("="*60)
test_counts = process_patients(test_patients, 'Test')

print("\n--- TEST ---")
for ch in [16, 8]:
    print(f"{ch} canales: bckg={test_counts[ch]['bckg']}, seiz={test_counts[ch]['seiz']}")

GENERANDO TEST


Test: 100%|██████████| 9/9 [24:01<00:00, 160.22s/it]


--- TEST ---
16 canales: bckg=1299, seiz=2402
8 canales: bckg=1299, seiz=2402


In [12]:
print("="*60)
print("RESUMEN FINAL - TUH EEG SEIZURE CORPUS")
print("="*60)

print(f"\nPacientes Training: {len(train_patients)}")
print(f"Pacientes Test: {len(test_patients)}")

print("\n--- TRAINING ---")
for ch in [16, 8]:
    total = train_counts[ch]['bckg'] + train_counts[ch]['seiz']
    print(f"{ch} canales: bckg={train_counts[ch]['bckg']}, seiz={train_counts[ch]['seiz']}, Total={total}")

print("\n--- TEST ---")
for ch in [16, 8]:
    total = test_counts[ch]['bckg'] + test_counts[ch]['seiz']
    print(f"{ch} canales: bckg={test_counts[ch]['bckg']}, seiz={test_counts[ch]['seiz']}, Total={total}")

print("\n" + "="*60)
print(f"Matrices guardadas en: {OUTPUT_DIR}")
print("="*60)

RESUMEN FINAL - TUH EEG SEIZURE CORPUS

Pacientes Training: 34
Pacientes Test: 9

--- TRAINING ---
16 canales: bckg=7482, seiz=9383, Total=16865
8 canales: bckg=7482, seiz=9383, Total=16865

--- TEST ---
16 canales: bckg=1299, seiz=2402, Total=3701
8 canales: bckg=1299, seiz=2402, Total=3701

Matrices guardadas en: /home/manu/TFG2/EEG-Gold-Standard-main/ConnectivityMatrices_TUH
